# RQ1 — Annotation sample, agreement, and classifier stability

Reproduces the sections *Annotation Procedure* and *Analysis* of the paper and the classifier
validation figures used in *Threats to Validity*:

| Result in the paper | Where |
|---|---|
| 487 repositories with 2+ files → frame of 149 → sample of 47 repositories (97 files) | section 1 |
| κ between the two annotators (mean over the 13 categories); κ of `ai_rules` | section 2 |
| κ of the classifier in the validation on the sample (κ = 0.79) | section 3 |
| Stability across the three classifier runs (agreement) | section 4 |

Inputs:
- `data/collection/detection/detection_top10000_2026-05-07.csv`
- `data/rq02/rq02_frame_149.csv` (frozen frame; see `data/rq02/README.md`)
- `data/rq02/rq02_sample_47.csv` (output of `scripts/sample_manual_validation.py`)
- `data/rq01/manual_annotation/rq01_manual_labels.csv` (annotator A, annotator B, and consensus)
- `data/rq01/classifier/` (validation on the sample and runs 1 to 3)

**Unit of κ.** Cohen's κ is computed per category (0/1 presence in each file) and summarized by the
mean over the 13 categories. Six files of the sample have no consensus label and are left out of
the comparisons, which use 91 files.

In [1]:
import math
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "data" / "collection").exists():
    ROOT = ROOT.parent

DETECTION = ROOT / "data/collection/detection/detection_top10000_2026-05-07.csv"
FRAME_149 = ROOT / "data/rq02/rq02_frame_149.csv"
SAMPLE_47 = ROOT / "data/rq02/rq02_sample_47.csv"
LABELS = ROOT / "data/rq01/manual_annotation/rq01_manual_labels.csv"
CLASSIFIER = ROOT / "data/rq01/classifier"
OUT = ROOT / "results/rq01"
OUT.mkdir(parents=True, exist_ok=True)

FORMATS = ["AGENTS.md", "CLAUDE.md", "COPILOT", "GEMINI.md", "CURSOR"]
CATEGORIES = [
    "run_and_build", "project_overview", "doc_metadata", "test_instructions",
    "project_structure", "system_design", "contribution_guidelines", "code_conventions",
    "ai_rules", "performance_&_resource_management", "troubleshooting", "security",
    "deployment_&_infrastructure",
]


def kappa(a, b):
    """Cohen's κ for two binary vectors."""
    a, b = np.asarray(a), np.asarray(b)
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return float("nan") if pe == 1 else (po - pe) / (1 - pe)


def kappa_per_category(x, y):
    """`x` and `y`: aligned DataFrames (same files), one 0/1 column per category."""
    return pd.Series({c: kappa(x[c], y[c]) for c in CATEGORIES})

## 1. Frame and sample

The frame of 149 repositories comes from a computation made in May 2026 on the content read live
from GitHub (pairs with both files above 92.95 characters, the P35, and textual similarity of at
most 0.5; see `00_sampling_frame.ipynb`). That content was not stored, so the frame is used as frozen
data. The sample is drawn from the frame with Cochran's formula (90% confidence, 10% margin, seed 42).

In [2]:
det = pd.read_csv(DETECTION)
n_2plus = int((det[FORMATS].sum(axis=1) >= 2).sum())
frame = pd.read_csv(FRAME_149)
sample = pd.read_csv(SAMPLE_47)

z, e, N = 1.645, 0.10, frame["repo"].nunique()
n0 = z ** 2 * 0.25 / e ** 2
n = n0 / (1 + (n0 - 1) / N)
print(f"Repositories with 2+ files: {n_2plus}")
print(f"Frame: {N} repositories ({len(frame)} pairs)")
print(f"Cochran: n0 = {n0:.2f}; with finite population correction n = {n:.2f} → {math.ceil(n)}")
print(f"Sample: {sample['repo'].nunique()} repositories")

Repositories with 2+ files: 487
Frame: 149 repositories (185 pairs)
Cochran: n0 = 67.65; with finite population correction n = 46.74 → 47
Sample: 47 repositories


## 2. Agreement between the annotators

In [3]:
labels = pd.read_csv(LABELS, keep_default_na=False)
print(f"Annotated files: {len(labels)} in {labels['repo'].nunique()} repositories")


def binarize(column):
    sets = column.apply(lambda v: {c.strip() for c in v.split(",") if c.strip()})
    unknown = set().union(*sets) - set(CATEGORIES)
    assert not unknown, unknown
    return pd.DataFrame({c: sets.apply(lambda s: int(c in s)) for c in CATEGORIES})


with_consensus = labels[labels["consenso"].str.strip() != ""].reset_index(drop=True)
print(f"Files with consensus: {len(with_consensus)}")
A, B, C = (binarize(with_consensus[col]) for col in ["anotador_A", "anotador_B", "consenso"])

k_annot = kappa_per_category(A, B)
print(f"κ between annotators (mean over the 13 categories): {k_annot.mean():.3f}")
print(f"κ of ai_rules: {k_annot['ai_rules']:.3f}")
k_annot.sort_values().round(3)

Annotated files: 97 in 47 repositories
Files with consensus: 91
κ between annotators (mean over the 13 categories): 0.717
κ of ai_rules: 0.548


ai_rules                             0.548
deployment_&_infrastructure          0.552
contribution_guidelines              0.615
project_overview                     0.623
system_design                        0.675
doc_metadata                         0.681
run_and_build                        0.691
troubleshooting                      0.715
security                             0.766
performance_&_resource_management    0.808
code_conventions                     0.848
test_instructions                    0.889
project_structure                    0.905
dtype: float64

## 3. Classifier validation on the sample

Before classifying the 1,828 files, the classifier was applied to the 97 files of the sample
(`rq01_validation_run.csv`), and its labels were compared with the annotators' consensus on the 91
files that have a consensus. The κ = 0.79 reported in the paper comes from this validation.

In [4]:
def load_classification(path):
    d = pd.read_csv(path)
    d["key"] = d["repo"].str.strip().str.lower() + "|" + d["arquivo_analisado"].str.strip()
    for c in CATEGORIES:
        d[c] = d[c].fillna(0).astype(int)
    return d.set_index("key")


keys = (with_consensus["repo"].str.lower() + "|" + with_consensus["arquivo"]).values
validation = load_classification(CLASSIFIER / "rq01_validation_run.csv").loc[keys, CATEGORIES].reset_index(drop=True)
k_val = kappa_per_category(C, validation)
k_val.to_csv(OUT / "classifier_validation_kappa.csv", header=["kappa"])
print(f"κ of the classifier in the validation (mean over the 13 categories, {len(C)} files): {k_val.mean():.2f}")
k_val.sort_values().round(3)

κ of the classifier in the validation (mean over the 13 categories, 91 files): 0.79


ai_rules                             0.607
doc_metadata                         0.716
contribution_guidelines              0.729
project_overview                     0.758
system_design                        0.768
code_conventions                     0.785
run_and_build                        0.802
deployment_&_infrastructure          0.822
performance_&_resource_management    0.834
troubleshooting                      0.839
security                             0.839
project_structure                    0.849
test_instructions                    0.961
dtype: float64

## 4. Stability across the three runs

- **mean agreement**: % of files with the same answer (0/1), averaged over the three pairs of runs
  and the 13 categories;
- **unanimous decisions**: % of the decisions (file × category) that are equal in the three runs;
- **identical files**: % of files with the same set of categories in the three runs.

In [5]:
runs = [load_classification(CLASSIFIER / f"rq01_classification_run_{i}.csv") for i in (1, 2, 3)]
idx = runs[0].index
assert all(set(x.index) == set(idx) for x in runs) and len(idx) == 1828
M = [x.loc[idx, CATEGORIES] for x in runs]

agreement = np.mean([(M[a] == M[b]).mean().mean() for a, b in combinations(range(3), 2)])
unanimous = ((M[0] == M[1]) & (M[1] == M[2])).values.mean()
identical = ((M[0] == M[1]) & (M[1] == M[2])).all(axis=1).mean()

stability = pd.Series({"mean_agreement_pct": 100 * agreement, "unanimous_decisions_pct": 100 * unanimous,
                       "identical_files_pct": 100 * identical})
stability.to_csv(OUT / "run_stability.csv", header=["value"])
stability.round(3)

mean_agreement_pct         83.504
unanimous_decisions_pct    75.257
identical_files_pct        14.333
dtype: float64